# Эксперимент: Wav2Vec2-base (frozen) + SVM

**Модель:** Wav2Vec2-base (frozen) → mean+std+max → SVM RBF  
**Группа:** 03_pretrained_frozen  
**Ориентир:** checkpoint_3/exp_11 — F1-bad=0.617, ROC-AUC=0.791

## Исправление по сравнению с checkpoint_3/exp_11

В оригинале: encoder запускался каждую эпоху → 9566 секунд.  
Здесь: SVM на предагрегированных эмбеддингах (аналогично exp_whisper_svm).  
Это также добавляет 5-fold CV и оптимизацию порога, которые отсутствовали в оригинале.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import mlflow
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from transformers import Wav2Vec2Model, Wav2Vec2Processor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds, get_durations, load_audio, get_durations
from shared.results_utils import save_result_csv
from shared.mlflow_utils import start_run, log_cv_fold

/home/dk/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "facebook/wav2vec2-base"
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
w2v_proc  = Wav2Vec2Processor.from_pretrained(MODEL_ID)
w2v_model = Wav2Vec2Model.from_pretrained(MODEL_ID).to(device)
w2v_model.eval()

def extract_wav2vec(path):
    y, _ = load_audio(path, sr=16000)
    inp = w2v_proc(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        hs = w2v_model(inp.input_values.to(device)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 19565.75it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_holdout_split()
print(f"Train+Val: {len(paths_trainval)}  |  Test: {len(paths_test)}")

Train+Val: 2359  |  Test: 417


In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
}

with start_run("exp_wav2vec_svm", group="03_pretrained_frozen"):

    mlflow.log_params({
        "encoder":        MODEL_ID,
        "encoder_frozen": True,
        "aggregation":    "mean+std+max",
        "embed_dim":      2304,
        "classifier":     "SVM RBF",
        "cv_folds":       config.CV_N_SPLITS,
        "class_weight":   "balanced",
        "note":           "checkpoint_3/exp_11 had DL head (9566s), switched to SVM for consistency",
    })

    fold_results = []
    t0 = time.perf_counter()

    for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
            get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

        print(f"\nФолд {fold+1}/{config.CV_N_SPLITS}")
        X_tr  = build_feature_matrix(paths_tr,  extract_wav2vec, n_jobs=1)
        X_val = build_feature_matrix(paths_val, extract_wav2vec, n_jobs=1)
        X_tr  = np.hstack([X_tr,  letters_tr,  get_durations(paths_tr).reshape(-1, 1)])
        X_val = np.hstack([X_val, letters_val, get_durations(paths_val).reshape(-1, 1)])

        grid = GridSearchCV(pipeline, param_grid, cv=3,
                            scoring="f1_macro", refit=True, n_jobs=-1)
        grid.fit(X_tr, labels_tr)
        clf = grid.best_estimator_
        print(f"  best={grid.best_params_}")

        y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
        thr = find_optimal_threshold(labels_val, y_proba_val)
        metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=True)
        fold_results.append(metrics)
        log_cv_fold(fold, f1_bad=metrics["f1_bad"], f1_macro=metrics["f1_macro"],
                    roc_auc=metrics["roc_auc"], threshold=thr)

    train_time_sec = time.perf_counter() - t0
    cv_agg = evaluate_cv_folds(fold_results)
    print_cv_summary(cv_agg)

    print("\nФинальная модель на train+val...")
    X_trainval = build_feature_matrix(paths_trainval, extract_wav2vec, n_jobs=1)
    X_test     = build_feature_matrix(paths_test,     extract_wav2vec, n_jobs=1)
    X_trainval = np.hstack([X_trainval, letters_trainval, get_durations(paths_trainval).reshape(-1, 1)])
    X_test     = np.hstack([X_test,     letters_test,     get_durations(paths_test).reshape(-1, 1)])

    final_grid = GridSearchCV(pipeline, param_grid, cv=5,
                              scoring="f1_macro", refit=True, n_jobs=-1)
    final_grid.fit(X_trainval, labels_trainval)
    mlflow.log_params({f"best_{k}": v for k, v in final_grid.best_params_.items()})

    cv_threshold = cv_agg["threshold_mean"]
    y_proba_test = final_grid.best_estimator_.predict_proba(X_test)[:, config.CLASS_BAD]
    test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)

    save_result_csv(
        exp_dir=exp_dir, experiment_id="exp_wav2vec_svm",
        experiment_name="Wav2Vec2-base (frozen) + SVM",
        model="Wav2Vec2-base (frozen) + SVM RBF",
        accuracy=test_metrics["accuracy"], f1_macro=test_metrics["f1_macro"],
        f1_bad=test_metrics["f1_bad"],     roc_auc=test_metrics["roc_auc"],
        precision_bad=test_metrics["precision_bad"], recall_bad=test_metrics["recall_bad"],
        threshold=test_metrics["threshold"],
        cv_f1_bad_std=cv_agg["f1_bad_std"], cv_f1_macro_std=cv_agg["f1_macro_std"],
        notes=f"5-fold CV + holdout | best={final_grid.best_params_} | thr={cv_threshold:.2f}",
        train_time_sec=train_time_sec,
    )
    print("Результаты сохранены")

df_folds = pd.DataFrame(fold_results)
df_folds.index = [f"Fold {i+1}" for i in range(len(fold_results))]
display(df_folds[["accuracy", "f1_macro", "f1_bad", "roc_auc", "threshold"]])


Фолд 1/5


/home/dk/.local/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.83      0.80      0.81       318
         bad       0.61      0.66      0.63       154

    accuracy                           0.75       472
   macro avg       0.72      0.73      0.72       472
weighted avg       0.76      0.75      0.75       472

Threshold : 0.38
Accuracy  : 0.7500
F1-macro  : 0.7211
F1-bad    : 0.6312
ROC-AUC   : 0.7852

Фолд 2/5
  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
              precision    recall  f1-score   support

        good       0.83      0.77      0.80       319
         bad       0.58      0.67      0.62       153

    accuracy                           0.74       472
   macro avg       0.71      0.72      0.71       472
weighted avg       0.75      0.74      0.74       472

Threshold : 0.36
Accuracy  : 0.7373
F1-macro  : 0.7111
F1-bad    : 0.6242
ROC-AUC   : 0.7693

Фолд 3/5
  best={'clf__C': 2.0, 'clf__gamma': 'scale'}
  

,accuracy,f1_macro,f1_bad,roc_auc,threshold
Fold 1,0.750000,0.721074,0.631250,0.785183,0.38
Fold 2,0.737288,0.711144,0.624242,0.769336,0.36
Fold 3,0.707627,0.699419,0.649746,0.806544,0.25
Fold 4,0.790254,0.763406,0.683706,0.823417,0.39
Fold 5,0.796178,0.779089,0.717647,0.854801,0.35
